**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [5]:
import pandas as pd
import os
import re
import multiprocessing
cores = multiprocessing.cpu_count()
from collections import Counter
from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

# Tokenizer
from nltk.tokenize import word_tokenize

# Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

def precision_macro(y_true, y_pred):
    return precision_score(y_true, y_pred, average='macro', zero_division=0)

def recall_macro(y_true, y_pred):
    return recall_score(y_true, y_pred, average='macro', zero_division=0)

def f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

scoring = {
                'accuracy': 'accuracy',
                'precision_macro': make_scorer(precision_macro),
                'recall_macro': make_scorer(recall_macro),
                'f1_macro': make_scorer(f1_macro)
            }

# Naive Bayes
from sklearn.naive_bayes import ComplementNB # "Particularly suited for imbalanced data sets" (sklearn documentation)
from sklearn.naive_bayes import MultinomialNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique
from sklearn.linear_model import LogisticRegression, Perceptron

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Validation croisée - 5-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

**Charger les données nettoyées**

In [6]:
folder = '../1-data/training_datasets'
datasets = os.listdir(folder)
datasets

['train_dataset_10pc.xlsx',
 'train_dataset_20pc.xlsx',
 'train_dataset_30pc.xlsx',
 'train_dataset_40pc.xlsx',
 'train_dataset_50pc.xlsx',
 'train_dataset_60pc.xlsx',
 'train_dataset_70pc.xlsx',
 'train_dataset_80pc.xlsx',
 'train_dataset_90pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [7]:
def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

In [8]:
# Load the test dataset
df_test = pd.read_excel('../1-data/test_dataset_10.xlsx')
corpus_test = df_test['text_post'].astype('str')
y_test = df_test['category'].astype('str')

training_report = []
test_report = []

for dataset in datasets:
    ratio = int(dataset[14:-7])
    df = pd.read_excel(os.path.join(folder, dataset))
    report = []

    corpus = df['text_post'].astype('str')
    y = df['category'].astype('str')

    # Values to test for the number of features
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]


    # Algorithms to test
    algorithms = {
        'Complement Naive Bayes' : ComplementNB(),
        'Multinomial Naive Bayes' : MultinomialNB(), 
        'kNN (k=5)' : KNeighborsClassifier(n_neighbors=5, n_jobs = cores),
        'kNN (k=10)' : KNeighborsClassifier(n_neighbors=10, n_jobs = cores),
        'kNN (k=15)' : KNeighborsClassifier(n_neighbors=15, n_jobs = cores),
        'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        'Perceptron': Perceptron(),
        'Random Forest': RandomForestClassifier(n_jobs = cores),
    }

    for n_features in n_features_values:
        tf_idf_vectorizer = TfidfVectorizer(
            stop_words=stopw,
            tokenizer=word_tokenize,
            token_pattern=None,
            max_features=n_features,
        )
        X = tf_idf_vectorizer.fit_transform(corpus)
        X_test = tf_idf_vectorizer.transform(corpus_test)

        for algorithm_name, algorithm in algorithms.items():
            # Perform cross-validation
            scores = cross_validate(
                estimator=algorithm,
                X=X,
                y=y,
                cv=stratified_kfold,
                scoring=scoring,
                return_train_score=False
            )

            # Collect results
            results = {
                '% Incels': int(ratio),
                'Algorithme': algorithm_name,
                'Modèle de vectorisation': 'TF-IDF',
                'N traits discr.': n_features,
                'accuracy': scores['test_accuracy'].mean(),
                'precision': scores['test_precision_macro'].mean(),
                'recall': scores['test_recall_macro'].mean(),
                'f1-score': scores['test_f1_macro'].mean()
            }

            report.append(results)

            print(algorithm_name, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n',
                  f"Accuracy: {results['accuracy']:.4f}, "
                  f"Precision: {results['precision']:.4f}, "
                  f"Recall: {results['recall']:.4f}, "
                  f"F1-score: {results['f1-score']:.4f}")
            clear_output(wait=True)

            # Fit the algorithm on the entire training set and evaluate on the test set
            # algorithm.fit(X, y)  # This does not involve cross-validation; it's a final fitting
            # predictions_test = algorithm.predict(X_test)

            # test_results = {
            #     '% Incels': int(ratio),
            #     'Algorithme': algorithm_name,
            #     'Modèle de vectorisation': 'TF-IDF',
            #     'N traits discr.': n_features,
            #     'accuracy': accuracy_score(y_test, predictions_test),
            #     'precision': precision_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'recall': recall_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'f1-score': f1_score(y_test, predictions_test, average='macro', zero_division=0)
            # }

            # test_report.append(test_results)

    # Export cross-validation results
    report_df = pd.DataFrame(report)
    report_df['nb_posts_incels'] = (report_df['% Incels'].apply(lambda x: x / 100) * df.shape[0]).astype(int)
    report_df['nb_posts_neutral'] = (report_df['% Incels'].apply(lambda x: 1 - (x / 100)) * df.shape[0]).astype(int)
    report_df['nb_posts_total'] = df.shape[0]

    # Reorganize column order
    report_df = report_df[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme',
                           'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall', 'f1-score']]

    training_report.append(report_df)

# Combine all cross-validation results into a single DataFrame
final_report_df = pd.concat(training_report)

# Save the final cross-validation report sorted by f1-score
final_report_df.sort_values(by='f1-score', ascending=False).to_csv(
    '../3-results/results_training_tfidf.csv', index=False
)

# Combine all test results into a single DataFrame
test_report_df = pd.DataFrame(test_report)

# Save the test results report sorted by f1-score
# test_report_df.sort_values(by='f1-score', ascending=False).to_csv(
#     '../3-results/results_test.csv', index=False
# )

Multinomial Naive Bayes  | 10% Incels |  500 traits discr.
 Accuracy: 0.9018, Precision: 0.8696, Recall: 0.5109, F1-score: 0.4958


**Nouvelles données** *(Valider que les données test n'ont jamais été vues)*

*Données test*

In [ ]:
file_test = '../1-data/test_dataset_10.xlsx'

df_test = pd.read_excel(file_test)
df_test

*Fichier 40% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_40pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]

*Fichier 50% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_20pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]

*Fichier 60% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_60pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]